# Bézier Tile Studio

A notebook about **Bézier tiles**. The Truchet notebook shows that tiles join up when their curves end on shared edge points. This notebook looks at the curve itself and asks three questions: what does a Bézier curve actually do, why is it the right tool for joining tiles, and how far can a tile library be pushed once it is **generated by math** instead of typed by hand?

| # | Step | What you see |
|---|------|--------------|
| 1 | Anatomy of a Bézier | One curve, its control points, and the `strength` dial |
| 2 | Why the normal matters | Two tiles that join smoothly, and the same two with a visible kink |
| 3 | A tile library from math | `K` points per edge, every non-crossing pairing, and how fast the library grows |
| 4 | Curvature as a dial, and its limit | The same tiles at low, medium and high strength, strength from a field, and when curves start to cross |
| 5 | Ordering the library | Tiles ranked from tight curls to long sweeps, then picked by a gradient and by a field |
| 6 | The final piece | Strands whose color flows continuously across tile edges |

Run the cells from top to bottom. It helps to have seen `demo_truchet.ipynb` first: this notebook reuses its idea of tiles written as text and its recipe of smooth random fields.

*Note:* the cells call `cubic_bezier(..., max_offset=...)`, so they need a version of `math_vis` where that parameter exists.

## 0. Setup and helpers

Imports and a few small helpers (including a check for curves that cross each other, which step 4 needs). Run this cell first. `make_tile_points` generalizes `P_TILE` and `N_TILE` from `math_vis` (two points per edge) to any number of points per edge, and `rotate_pairs` generalizes `rotate_tile` in the same way.

In [ ]:
%matplotlib inline
import itertools
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import ListedColormap

from math_vis import (
    make_cmap, smooth_field, sample, squash, finish, fig_to_img, new_rng,
    cubic_bezier, matchings, P_TILE, N_TILE,
)

NAVY, CORAL = "#1d3557", "#c8553d"

def make_tile_points(k):
    """Boundary points of a unit tile with k points per edge, and their inward normals.

    The 4k points are numbered counter-clockwise from the bottom left: bottom edge
    left to right, right edge bottom to top, top edge right to left, left edge top
    to bottom. The positions are symmetric about the middle of the edge, so every
    edge carries the same (palindromic) pattern and neighbouring tiles always agree.
    """
    pos = 0.5 + (np.arange(k) - (k - 1) / 2) * (0.8 / k)
    zero, one = np.zeros(k), np.ones(k)
    P = np.vstack([np.column_stack([pos, zero]),         # bottom
                   np.column_stack([one, pos]),          # right
                   np.column_stack([1 - pos, one]),      # top
                   np.column_stack([zero, 1 - pos])])    # left
    N = np.repeat([[0.0, 1.0], [-1.0, 0.0], [0.0, -1.0], [1.0, 0.0]], k, axis=0)
    return P, N

def rotate_pairs(tile, r, k):
    """Rotate a tile (a list of (i, j) pairs) by r quarter turns: every label moves k places."""
    m = 4 * k
    return [((i + k * r) % m, (j + k * r) % m) for i, j in tile]

def bez(P, N, i, j, strength, max_offset=None, n=40):
    """Bezier curve between boundary points i and j, leaving both along the inward edge normal."""
    return cubic_bezier(P[i], P[j], n=n, strength=strength,
                        normal_p=N[i], normal_q=N[j], max_offset=max_offset)

def grid_curves(lib, tile_idx, turns, strength, P, N, k, max_offset=None, n=30):
    """All curves of a grid of tiles, as a list of (n, 2) arrays in grid coordinates.

    tile_idx : (NY, NX) which tile of `lib` sits in each cell
    turns    : (NY, NX) quarter turns per cell
    strength : a number, or an (NY, NX) array with one value per cell
    """
    ny, nx = tile_idx.shape
    strength = np.broadcast_to(strength, (ny, nx))
    curves = []
    for j in range(ny):
        for i in range(nx):
            for a, b in rotate_pairs(lib[tile_idx[j, i]], turns[j, i], k):
                curves.append(bez(P, N, a, b, strength[j, i], max_offset, n) + [i, j])
    return curves

def span(tile, k):
    """Average reach of the curves of a tile: small = curls near the edges, large = curves reaching across."""
    m = 4 * k
    ring = [min(abs(i - j), m - abs(i - j)) for i, j in tile]    # distance along the boundary, shorter way round
    return np.mean(ring) / (m / 2)                               # as a fraction of half the boundary

def polylines_cross(a, b):
    """True if the polylines a and b (arrays of points) cross each other."""
    a0, a1 = a[:-1, None, :], a[1:, None, :]
    b0, b1 = b[None, :-1, :], b[None, 1:, :]
    cross = lambda u, v: u[..., 0] * v[..., 1] - u[..., 1] * v[..., 0]
    b_straddles_a = cross(a1 - a0, b0 - a0) * cross(a1 - a0, b1 - a0) < 0
    a_straddles_b = cross(b1 - b0, a0 - b0) * cross(b1 - b0, a1 - b0) < 0
    return bool(np.any(b_straddles_a & a_straddles_b))

def tile_has_crossing(tile, P, N, strength, max_offset=None):
    """True if any two curves of the tile cross each other when drawn."""
    curves = [bez(P, N, i, j, strength, max_offset) for i, j in tile]
    return any(polylines_cross(a, b) for a, b in itertools.combinations(curves, 2))

def offset(c, d):
    """Shift a curve sideways by d, perpendicular to its direction."""
    tang = np.gradient(c, axis=0)
    tang /= np.linalg.norm(tang, axis=1, keepdims=True)
    return c + d * np.column_stack([-tang[:, 1], tang[:, 0]])

def to_index(field, n):
    """Split a field into n equally common bins (by quantiles) and return the bin index per cell."""
    if n == 1:
        return np.zeros(field.shape, dtype=int)
    cuts = np.quantile(field, np.linspace(0, 1, n + 1)[1:-1])
    return np.digitize(field, cuts)

---
## 1. Anatomy of a Bézier curve

**Principle.** A cubic Bézier curve is defined by four points: the start `p`, the end `q`, and two **control points** `c1` and `c2`. The curve leaves `p` heading toward `c1` and arrives at `q` coming from `c2`. It never touches the control points; it is only *pulled* toward them. As a formula, with `t` running from 0 to 1:

`B(t) = (1−t)³ p + 3(1−t)² t c1 + 3(1−t) t² c2 + t³ q`

`cubic_bezier` places the control points for you:

```
c1 = p + s · (normal at p)      c2 = q + s · (normal at q)      s = strength · distance(p, q)
```

So `strength` is how far the control points sit from the ends, as a fraction of the distance between the ends. Small `strength`: nearly a straight line. Large `strength`: the curve first bulges out along the normals and turns late.

**Recipe**
1. Choose `p`, `q`, and a normal at each: the direction the curve should leave in.
2. Choose several `strength` values.
3. Call `cubic_bezier`, and also draw the thin lines `p → c1` and `q → c2` so that the control points are visible.

Solid red dots are the end points, hollow dots are the control points, and the dashed square is the tile. The curve always starts exactly *along* the thin line.

In [ ]:
STRENGTHS = [0.15, 0.35, 0.55, 1.0]                      # the dial to compare
p, n_p = np.array([0.3, 0.0]), np.array([0.0, 1.0])      # start: bottom edge, normal points up into the tile
q, n_q = np.array([1.0, 0.7]), np.array([-1.0, 0.0])     # end: right edge, normal points left into the tile

fig, axs = plt.subplots(1, 4, figsize=(15, 4.2))
for ax, s in zip(axs, STRENGTHS):
    curve = cubic_bezier(p, q, n=60, strength=s, normal_p=n_p, normal_q=n_q, max_offset=None)
    reach = s * np.linalg.norm(p - q)                    # the distance "s" from the formula
    c1, c2 = p + reach * n_p, q + reach * n_q            # the two control points
    ax.plot([0, 1, 1, 0, 0], [0, 0, 1, 1, 0], color="#bbbbbb", lw=0.8, ls="--")
    ax.plot(*zip(p, c1), color="#999999", lw=1)
    ax.plot(*zip(q, c2), color="#999999", lw=1)
    ax.scatter(*zip(c1, c2), s=45, facecolor="white", edgecolor=CORAL, zorder=3)    # control points
    ax.scatter(*zip(p, q), s=45, color=CORAL, zorder=3)                              # end points
    ax.plot(curve[:, 0], curve[:, 1], color=NAVY, lw=3)
    ax.set_title(f"strength = {s}\ncontrol points sit {reach:.2f} from the ends")
    ax.set_aspect("equal"); ax.axis("off"); ax.set_xlim(-0.25, 1.25); ax.set_ylim(-0.25, 1.25)
plt.tight_layout(); plt.show()

**Things to try.** Pass `max_offset=0.2` to `cubic_bezier`: the control points are never pushed further than 0.2 from the ends, so every `strength` above about 0.3 now looks the same. That cap is what the classic Truchet tiles use, and it hides this dial. Here the cap is switched off with `max_offset=None`.

---
## 2. Why the normal matters

**Principle.** Two tiles side by side share an edge point. A curve crosses that edge smoothly only if the two curves have the **same direction** there. A Bézier curve leaves `p` in the direction of `c1`, so the direction at the edge is whatever `normal_p` says. If every tile uses the **edge normal**, all curves that touch the same point leave it in the same direction. They continue each other, whichever tile and rotation sits on the other side.

If `normal_p` and `normal_q` are left out, `cubic_bezier` aims the control points along the straight line toward the other end of the curve. Then both control points lie on the straight line between the ends, so the curve is just a straight line, and it ignores the neighbour: it leaves a **corner at the edge**. (Mathematically: the first panel is tangent-continuous, called G¹, across the shared point; the second is not.)

**Recipe**
1. Take two tiles, A on the left and B on the right. Their shared edge point is the red dot.
2. Draw A's curve arriving at the dot and B's curve leaving it, using the edge normals (left panel).
3. Draw the same two curves without `normal_p` and `normal_q` (right panel).
4. Measure the **kink**: the angle between the arriving and the leaving direction at the dot.

In [ ]:
def tile_pair(i, j, shift, use_normals, strength=0.55):
    """Curve between points i and j of the classic 8-point tile, moved by `shift`."""
    kw = dict(normal_p=N_TILE[i], normal_q=N_TILE[j]) if use_normals else {}
    return cubic_bezier(P_TILE[i], P_TILE[j], n=400, strength=strength, max_offset=0.3, **kw) + shift

def kink_deg(c_in, c_out):
    """Angle between the direction arriving at the joint and the direction leaving it."""
    d_in, d_out = c_in[-1] - c_in[-2], c_out[1] - c_out[0]
    cos = d_in @ d_out / (np.linalg.norm(d_in) * np.linalg.norm(d_out))
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

fig, axs = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, use_normals, label in zip(axs, [True, False], ["with the edge normals", "without (default directions)"]):
    a = tile_pair(0, 3, np.array([0.0, 0.0]), use_normals)      # tile A: bottom point 0 -> right point 3
    b = tile_pair(6, 1, np.array([1.0, 0.0]), use_normals)      # tile B: left point 6 -> bottom point 1
    for x0 in (0, 1):
        ax.plot([x0, x0 + 1, x0 + 1, x0, x0], [0, 0, 1, 1, 0], color="#bbbbbb", lw=0.8, ls="--")
    ax.plot(a[:, 0], a[:, 1], color=NAVY, lw=4, solid_capstyle="butt")
    ax.plot(b[:, 0], b[:, 1], color=CORAL, lw=4, solid_capstyle="butt")
    ax.scatter([1], [0.7], s=70, color="white", edgecolor="black", zorder=4)
    ax.set_title(f"{label}\nkink at the joint = {kink_deg(a, b):.0f}°")
    ax.set_aspect("equal"); ax.axis("off")
plt.tight_layout(); plt.show()

**Things to try.** Change the pairs, for example `tile_pair(0, 2, ...)` and `tile_pair(5, 1, ...)`. With the normals the kink is always 0°, whichever two points you pick: that is why *any* tile can sit next to *any* other tile in the library below. Without the normals every curve is a straight chord, and the kink depends on the geometry of the particular pair.

---
## 3. A tile library from math

**Principle.** In the Truchet notebook the tile library was typed by hand (`TILE_TEXT`) for two points per edge. But the only constraint that makes tiles joinable concerns the **edge points** (the same on all four edges, symmetric about the middle). The **interior is completely free**. So the library can be *generated*: choose `K` points per edge, number the `4K` boundary points around the tile, and list every way of pairing them up **without crossing**. `matchings` does exactly that.

The number of such pairings is a Catalan number: 2, 14, 132, 1430, ... for `K = 1, 2, 3, 4`. With `K = 2` you get the 14 tiles of the Truchet notebook. With `K = 3` there are already 132.

**Recipe**
1. `make_tile_points(K)` gives the `4K` boundary points and their inward normals, numbered counter-clockwise from the bottom left.
2. `matchings(list(range(4 * K)))` gives all non-crossing pairings. Each one is a tile, written as a list of `(i, j)` pairs.
3. Rotating a tile by `r` quarter turns means adding `K · r` to every label (modulo `4K`). For `K = 2` that is the "+2" of the Truchet notebook.
4. Draw each pair with `cubic_bezier`, leaving along the edge normals (step 2).

The first cell checks the generalization against `math_vis` and counts tiles. The second draws a random sample of the `K = 3` library.

In [ ]:
# Sanity check: for K = 2 the generated points are exactly the library's P_TILE and N_TILE
P2, N2 = make_tile_points(2)
print("K = 2 reproduces math_vis P_TILE / N_TILE:", np.allclose(P2, P_TILE) and np.allclose(N2, N_TILE))

# How fast does the library grow?
for k in range(1, 5):
    print(f"K = {k}:  {4 * k:2d} boundary points  ->  {len(matchings(list(range(4 * k)))):5d} tiles")

In [ ]:
SEED = None
K = 3                  # points per edge
STRENGTH = 0.5         # how round the curves are
MAX_OFFSET = 0.3       # a safety rail on the curves' reach; step 4 explains why it is needed
rng = new_rng(SEED)

P, N = make_tile_points(K)
LIB = matchings(list(range(4 * K)))                    # the whole library: every non-crossing pairing

# Rotating a non-crossing tile gives a non-crossing tile, so rotations stay inside the library.
as_set = lambda tile: frozenset(frozenset(pair) for pair in tile)
lib_sets = {as_set(t) for t in LIB}
print("rotations stay inside the library:", all(as_set(rotate_pairs(t, r, K)) in lib_sets for t in LIB for r in range(4)))

picks = rng.choice(len(LIB), size=12, replace=False)
fig, axs = plt.subplots(2, 6, figsize=(15, 5.6))
for ax, t in zip(axs.ravel(), picks):
    ax.plot([0, 1, 1, 0, 0], [0, 0, 1, 1, 0], color="#cccccc", lw=0.7, ls="--")
    for a, b in LIB[t]:
        c = bez(P, N, a, b, STRENGTH, MAX_OFFSET)
        ax.plot(c[:, 0], c[:, 1], color=NAVY, lw=2, solid_capstyle="round")
    ax.scatter(P[:, 0], P[:, 1], s=8, color=CORAL, zorder=3)
    ax.set_title(f"tile {t} of {len(LIB)}", fontsize=9); ax.set_aspect("equal"); ax.axis("off")
plt.suptitle(f"12 random tiles of the K = {K} library ({len(LIB)} tiles in total)", fontsize=12)
plt.tight_layout(); plt.show()

**Things to try.** `K = 2` shows the classic library again (14 tiles). `K = 4` has 1430 tiles; generating them takes about a second. `K = 1` has only two tiles, which are the two orientations of the classic arc tile, drawn as Béziers instead of circle arcs.

---
## 4. Curvature as a dial, and its limit

**Principle.** The edge points and normals fix **where** curves meet and **in which direction**. They do not fix `strength`. So every curve may have its own strength, and the joins stay smooth: the direction at the edge is unchanged, only how sharply the curve turns afterwards differs. A smooth random field for the strength makes the picture "breathe": calm and nearly straight in some regions, round and energetic in others.

**Recipe**
1. Make one random grid of tiles and rotations. This is shared by all four panels, so that **only the dial changes**.
2. Draw it with a constant low, medium and high strength.
3. Draw it again with strength from a smooth field: `strength = LOW + (HIGH − LOW) · squash(field)`.
4. Then look closely at the high-strength panels, and count what goes wrong.

In [ ]:
SEED = None
K = 3
NX, NY = 9, 6
LOW, HIGH = 0.15, 0.65          # the strength range
FIELD_SMOOTH = 1.5              # size of the regions with similar strength (in tiles)
rng = new_rng(SEED)

P, N = make_tile_points(K)
LIB = matchings(list(range(4 * K)))
tile_idx = rng.integers(len(LIB), size=(NY, NX))         # a random tile in every cell
turns    = rng.integers(4, size=(NY, NX))                # and a random rotation
field    = smooth_field((NY, NX), FIELD_SMOOTH, rng)
strength_map = LOW + (HIGH - LOW) * squash(field)        # one strength per tile, from the field

panels = [(f"strength = {LOW} everywhere", LOW),
          (f"strength = {(LOW + HIGH) / 2:.2f} everywhere", (LOW + HIGH) / 2),
          (f"strength = {HIGH} everywhere", HIGH),
          ("strength from a smooth field", strength_map)]

fig, axs = plt.subplots(2, 2, figsize=(13, 9.4))
for ax, (title, s) in zip(axs.ravel(), panels):
    curves = grid_curves(LIB, tile_idx, turns, s, P, N, K)
    ax.add_collection(LineCollection(curves, colors=NAVY, linewidths=1.6, capstyle="round"))
    ax.set_xlim(0, NX); ax.set_ylim(0, NY); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(title)
plt.tight_layout(); plt.show()

### A limit: curves can cross

Look closely at the `strength = 0.65` panel (and, if you look carefully, at 0.40): in several places two curves **cross each other**, although every tile is made of *non-crossing* pairs. The pairing is non-crossing in the sense of *which points are connected*. But a Bézier curve with long control arms bulges, and a bulge can reach across another curve. Non-crossing as a **pairing** (topology) is not the same as non-crossing as a **drawing** (geometry).

The remedy is the cap from step 1: `max_offset` limits how far the control points reach, however large `strength` is. It is the cap of 0.2 that the classic `tile_curve` uses. The cell below counts, for the whole `K = 3` library, the tiles that contain crossing curves at several strengths, without a cap and with a cap of 0.3.

In [ ]:
K = 3
STRENGTHS = [0.25, 0.35, 0.45, 0.55, 0.70]
CAPS = [None, 0.3]

P, N = make_tile_points(K)
LIB = matchings(list(range(4 * K)))
print(f"Tiles (out of {len(LIB)}) that contain crossing curves when drawn:\n")
print("strength   no cap   max_offset=0.3")
for s in STRENGTHS:
    counts = [sum(tile_has_crossing(t, P, N, s, cap) for t in LIB) for cap in CAPS]
    print(f"{s:8.2f}   {counts[0]:6d}   {counts[1]:14d}")

With the cap the count is 0 at every strength: the cap works as a safety rail. The price is that long curves all get the same reach, so `strength` only keeps shaping the short and medium curves. The steps below use `max_offset = 0.3`.

**Things to try.** `HIGH = 1.2` in the cell above: curves that cross the tile now bulge out of their own tile and overlap their neighbours, but they still join smoothly. `LOW = 0.02` makes nearly straight chords. `FIELD_SMOOTH = 0.01` gives every tile an independent strength. In the count, try `CAPS = [None, 0.4, 0.3, 0.2]` and `K = 2`: the classic 14 tiles only start to cross at higher strengths, which is why the old cap of 0.2 was safe.

---
## 5. Ordering the library

**Principle.** With 132 tiles you cannot choose by hand, and a random choice per cell gives noise. But you can **order** the library by a measurable property and then pick tiles by *rank*, using a gradient or a smooth field. Neighbouring cells then get tiles of similar character, and the picture drifts gradually from one kind of tile to another.

The property used here is the **span** of a tile. For each curve, take the distance along the boundary between its two ends (the shorter way round). The span of the tile is the average over its curves, as a fraction of half the boundary length. Small span: small curls that hug the edges. Large span: longer curves that reach across the tile. (Curves cannot all be long without crossing each other, so for `K = 3` the span only runs from about 0.17 to 0.5.)

**Recipe**
1. Compute `span` for every tile and sort the library by it.
2. Look at a strip of tiles at evenly spaced ranks (left = curls, right = sweeps).
3. Choose the rank per cell from (a) a plain left-to-right gradient and (b) a smooth random field. `to_index` turns a field into equally likely ranks, by quantiles, as in the Truchet notebook.

In [ ]:
K = 3
MAX_OFFSET = 0.3                    # the safety rail from step 4
P, N = make_tile_points(K)
LIB = sorted(matchings(list(range(4 * K))), key=lambda t: span(t, K))     # the ranked library
spans = np.array([span(t, K) for t in LIB])
print(f"{len(LIB)} tiles, span from {spans[0]:.2f} (curls) to {spans[-1]:.2f} (sweeps)")

ranks = np.linspace(0, len(LIB) - 1, 8).astype(int)
fig, axs = plt.subplots(1, 8, figsize=(16, 2.7))
for ax, r in zip(axs, ranks):
    ax.plot([0, 1, 1, 0, 0], [0, 0, 1, 1, 0], color="#cccccc", lw=0.7, ls="--")
    for a, b in LIB[r]:
        c = bez(P, N, a, b, 0.5, MAX_OFFSET)
        ax.plot(c[:, 0], c[:, 1], color=NAVY, lw=2, solid_capstyle="round")
    ax.set_title(f"rank {r}\nspan {spans[r]:.2f}", fontsize=9); ax.set_aspect("equal"); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
SEED = None
NX, NY = 18, 8
SMOOTH = 2.5                     # size of the regions in the random-field version (in tiles)
rng = new_rng(SEED)

turns = rng.integers(4, size=(NY, NX))                                   # same rotations in both panels
gradient = np.tile(np.linspace(0, 1, NX), (NY, 1))                       # 0 on the left, 1 on the right
rank_gradient = np.minimum((gradient * len(LIB)).astype(int), len(LIB) - 1)
rank_field = to_index(smooth_field((NY, NX), SMOOTH, rng), len(LIB))     # equally common ranks

fig, axs = plt.subplots(2, 1, figsize=(12.5, 11))
for ax, idx, title in zip(axs, [rank_gradient, rank_field],
                          ["rank from a left-to-right gradient: curls on the left, sweeps on the right",
                           "rank from a smooth random field: regions of curls and regions of sweeps"]):
    curves = grid_curves(LIB, idx, turns, 0.5, P, N, K, MAX_OFFSET)
    ax.add_collection(LineCollection(curves, colors=NAVY, linewidths=1.5, capstyle="round"))
    ax.set_xlim(0, NX); ax.set_ylim(0, NY); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(title)
plt.tight_layout(); plt.show()

**Things to try.** Sort by another property: the number of curves that stay on one edge, or how deeply the curves are nested. Reverse the order with `sorted(..., reverse=True)`. Use `K = 2` and see that 14 tiles give visible bands rather than a smooth drift, because there are too few ranks.

---
## 6. The final piece

**Principle.** Everything combined: tiles from the **generated, ranked library**, picked by a smooth field; rotation and strength from two more smooth fields; each curve drawn as parallel **strands**. One new idea: the color is **not one value per tile**. It comes from a smooth field that is *sampled along each curve* (`sample`), so the color changes gradually along every ribbon and flows across tile edges without a seam.

One detail makes this work. A strand's color may depend on *how far* the strand is from the middle of the ribbon, but not on *which side* it is on. Two neighbouring tiles can traverse the same ribbon in opposite directions, and then they disagree about which side is "left". Shifting the palette by `−abs(offset)` makes both outer strands the same, and the two tiles always agree.

**Recipe**
1. Four smooth random fields: rotation, tile rank and strength (one value per tile), and color (on a finer grid, `COLOR_RES` cells per tile).
2. Rank → a tile of the sorted library, rotation → quarter turns, strength → how round.
3. Build every curve with `cubic_bezier` and move it to its grid cell.
4. Color: sample the color field along the curve, then `palette(squash(value) − strand_shift)`.
5. Draw all strands as short colored segments.

**Look at the randomness first.** The first cell plots the four fields before any drawing, so you can see which decisions the artwork is made from, and then checks that no curves cross. The second cell draws it.

In [ ]:
# =====================================================================
# SETTINGS
# =====================================================================
K               = 3            # points per tile edge: 3 -> 132 tiles, 2 -> the classic 14
SEED            = 7            # None = new result every run; a number = repeatable
NX, NY          = 12, 16       # tiles across and down
TILE_WINDOW     = (0.0, 1.0)   # part of the ranked library to use: (0, 0.3) = curls only, (0.7, 1) = sweeps only
STRENGTH_RANGE  = (0.2, 0.7)   # how round the curves get
MAX_OFFSET      = 0.3          # cap on the control-point distance (tile = 1.0); None = no cap (curves may cross)
PICK_SMOOTH     = 2.5          # size of the regions with similar tiles (in tiles)
TURN_SMOOTH     = 1.5          # size of the regions with the same rotation
STRENGTH_SMOOTH = 2.5          # size of the regions with similar strength
COLOR_SMOOTH    = 3.0          # size of the color regions (in tiles)
COLOR_RES       = 4            # color-field cells per tile
N_STRANDS       = 3            # parallel strands per curve
BAND            = 0.045        # spacing between strands (tile = 1.0)
STRAND_SHIFT    = 0.10         # how far the outer strands drift along the palette
BG              = "#f1e9d2"
DPI             = 120          # resolution of the final image (use 300 for a print-size file)

rng = new_rng(SEED)
palette = make_cmap(["#d9674a", "#e8a56b", "#e6d28f", "#9db08c", "#5f8a86", "#2c3a35"])

P, N = make_tile_points(K)
LIB = sorted(matchings(list(range(4 * K))), key=lambda t: span(t, K))
lo, hi = int(TILE_WINDOW[0] * len(LIB)), int(TILE_WINDOW[1] * len(LIB))
tiles = LIB[lo:hi]                                                   # the part of the library in use

# ---------- FOUR INDEPENDENT SMOOTH FIELDS ----------
turn_field     = smooth_field((NY, NX), TURN_SMOOTH, rng)            # decides rotation
pick_field     = smooth_field((NY, NX), PICK_SMOOTH, rng)            # decides which tile
strength_field = smooth_field((NY, NX), STRENGTH_SMOOTH, rng)        # decides how round
color_field    = smooth_field((NY * COLOR_RES, NX * COLOR_RES), COLOR_SMOOTH * COLOR_RES, rng)   # decides color

turns        = to_index(turn_field, 4)
tile_idx     = to_index(pick_field, len(tiles))
strength_map = STRENGTH_RANGE[0] + (STRENGTH_RANGE[1] - STRENGTH_RANGE[0]) * squash(strength_field)

# ---------- PLOT THE RANDOMNESS FIRST ----------
fig, axs = plt.subplots(1, 4, figsize=(15, 4.6))
turn_cmap = ListedColormap(["#1d3557", "#5f8a86", "#e6d28f", "#d9674a"])
maps = [(turns, turn_cmap, "quarter turns (0 to 3)"),
        (tile_idx, "viridis", f"tile rank (0 = curls, {len(tiles) - 1} = sweeps)"),
        (strength_map, "magma", "strength (how round)"),
        (color_field, "RdBu_r", "color field (finer grid)")]
for ax, (m, cmap, title) in zip(axs, maps):
    ax.imshow(m, origin="lower", cmap=cmap, interpolation="nearest"); ax.set_title(title, fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()

# ---------- SAFETY CHECK: do any curves cross at the highest strength? ----------
n_cross = sum(tile_has_crossing(t, P, N, STRENGTH_RANGE[1], MAX_OFFSET) for t in tiles)
print(f"tiles with crossing curves at the highest strength: {n_cross} of {len(tiles)}")

In [ ]:
# ---------- BUILD EVERY CURVE ----------
curves = grid_curves(tiles, tile_idx, turns, strength_map, P, N, K, MAX_OFFSET, n=24)
offsets = np.linspace(-1, 1, N_STRANDS) if N_STRANDS > 1 else np.array([0.0])

segments, colors = [], []
for base in curves:
    # THE COLOR STEP: sample the color field along the curve (rows = y, columns = x), squash to 0..1
    t = squash(sample(color_field, base[:, 1] * COLOR_RES, base[:, 0] * COLOR_RES))
    t = 0.5 * (t[:-1] + t[1:])                                       # one value per little segment
    for off in offsets:
        c = offset(base, off * BAND * (N_STRANDS - 1) / 2)
        segments.append(np.stack([c[:-1], c[1:]], axis=1))
        colors.append(palette(np.clip(t - abs(off) * STRAND_SHIFT, 0, 1)))   # depends on |off|, not its sign

# ---------- DRAW ----------
STRAND_W = BAND * (8 * 72 / NX) * 0.9                                # line width in points that fills the spacing
fig = plt.figure(figsize=(8, 8 * NY / NX))
ax = fig.add_axes([0, 0, 1, 1])
fig.patch.set_facecolor(BG)
ax.set_xlim(0, NX); ax.set_ylim(0, NY); ax.axis("off")
ax.add_collection(LineCollection(np.concatenate(segments), colors=np.concatenate(colors),
                                 linewidths=STRAND_W, capstyle="round", antialiased=True))
finish(fig_to_img(fig, dpi=DPI), "bezier_tiles.png", grain=3)

---
## Cheat sheet

| Idea | Where | Takeaway |
|------|-------|----------|
| Control points `p + s·normal` | Step 1 | `strength` is how far the control points reach, as a fraction of the distance between the ends |
| Edge normal as direction | Step 2 | The same direction at a shared point means no kink between tiles (G¹ continuity) |
| Generated library | Step 3 | `4K` boundary points give a Catalan number of tiles: 14, 132, 1430, ... |
| Strength per tile | Step 4 | A free parameter: it changes the shape inside a tile, never the joins |
| Non-crossing pairs can still cross | Step 4 | Topology is not geometry: long control arms bulge, so cap them with `max_offset` |
| Ranking by span | Step 5 | Sorting a large library lets a smooth field change the character of the tiles gradually |
| Color sampled along curves | Step 6 | The color flows across tile edges; strand shifts may depend on the distance from the ribbon centre, not on its side |

**Experiments that teach the most**
- Step 1: compare `max_offset=None` with `max_offset=0.2` for the same strengths.
- Step 2: pick other pairs in `tile_pair` and check that the kink stays 0° with normals.
- Step 3: set `K = 4` and look at how many more tiles there are, and how few you can tell apart at a glance.
- Step 4: try `HIGH = 1.2` and watch curves leave their tiles, then extend the crossing count with `CAPS = [None, 0.4, 0.3, 0.2]`.
- Step 5: sort by something else, or use `K = 2` and see the bands.
- Step 6: use `TILE_WINDOW = (0, 0.25)` and then `(0.75, 1)`; try `N_STRANDS = 5` with a smaller `BAND`; set `COLOR_SMOOTH = 0.5` for speckled color; set `STRAND_SHIFT` negative for dark edges.

**If it looks wrong**
- *Strands collide in tight curls:* lower `BAND` or `N_STRANDS`.
- *Curves cross or overlap their neighbours:* lower `MAX_OFFSET` (the safety check prints the number of tiles with crossings).
- *Looks like static:* raise `PICK_SMOOTH` and `TURN_SMOOTH`.
- *Looks flat:* widen `TILE_WINDOW`, widen `STRENGTH_RANGE`, or lower `COLOR_SMOOTH`.

## Further reading

- Conceptual background: [Patterns built from tiles](conceptualGuide.md#4-patterns-built-from-tiles) in `conceptualGuide.md`.
- Implementation details: [Truchet weave](recipes.md#5-truchet-weave) in `recipes.md`.
- The building blocks used here: `cubic_bezier` and `matchings` in `math_vis/math_tools.py`.